# Milestone 5

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
LABELS = ['A','B','C','D','E']
label2idx = {l:i for i,l in enumerate(LABELS)}
idx2label = {i:l for i,l in enumerate(LABELS)}

## Setup: Load Fine-Tuned Models

In [ ]:
# Load fine-tuned checkpoints
deberta_model = AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=5)
deberta_tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')

roberta_model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=5)
roberta_tok = AutoTokenizer.from_pretrained('roberta-base')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
deberta_model = deberta_model.to(device).eval()
roberta_model = roberta_model.to(device).eval()

In [ ]:
def get_probs(model, tokenizer, prompt, options, device):
    """Get softmax probabilities for each option."""
    texts = [str(prompt) + ' [SEP] ' + str(opt) for opt in options]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits.squeeze(), dim=-1)
    return probs.cpu().numpy()